# SplitFed Demonstration Notebook

This notebook provides a simple interactive setup of the Split Federated Learning (SplitFed) simulator.
We will load the MNIST dataset, partition it among 3 simulated clients, and run a short training loop.

### Step 0: Remote Server Setup

If you are running this notebook on a remote Jupyter server (like Google Colab), run the following cell to clone the repository, install dependencies, and set the correct working directory.

In [ ]:
import os

# Clone the repository if it doesn't exist
if not os.path.exists('ad-sfl'):
    !git clone https://github.com/tomal66/ad-sfl.git

# Change directory to the repository root so imports work correctly
if os.path.basename(os.getcwd()) != 'ad-sfl':
    os.chdir('ad-sfl')

# Install requirements (uncomment if needed)
!pip install -r requirements.txt -q

print("Current working directory:", os.getcwd())

In [ ]:
import torch
from src.data.datasets import get_datasets
from src.data.partition import partition_data_iid
from src.models.split import ClientModel, ServerModel
from src.core.client import SplitFedClient
from src.core.server import SplitFedServer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Step 1: Load and Partition Data

In [ ]:
num_clients = 3
batch_size = 64

print("Loading MNIST dataset...")
train_data, test_data = get_datasets("MNIST")

print("Partitioning data info IID subsets...")
client_datasets = partition_data_iid(train_data, num_clients)
print(f"Data partitioned to {len(client_datasets)} clients.")

### Step 2: Initialize Server and Client Models

In SplitFed, a portion of the network is on the client, and the rest is on the server.

In [ ]:
import copy

# Server Model initialization
server_model = ServerModel(in_channels=32, hidden_channels=64, num_classes=10, input_size=(14, 14))
server = SplitFedServer(model=server_model, num_clients=num_clients, device=device)

# Client Models initialization (each client starts with same weights)
base_client_model = ClientModel(in_channels=1, hidden_channels=32)
clients = []
for i in range(num_clients):
    client_model = copy.deepcopy(base_client_model)
    client = SplitFedClient(client_id=i, model=client_model, dataset=client_datasets[i], batch_size=batch_size, device=device)
    clients.append(client)

print("Initialized 1 Server and 3 Clients.")

### Step 3: Simulation Loop

In [ ]:
epochs = 2

for epoch in range(epochs):
    epoch_loss = 0.0
    
    # Note: A real simulation might iterate multiple batches per client per round.
    # Here, for simplicity, we step through each client sequentially for a single batch.
    for client in clients:
        # 1. Client forward pass to cut layer
        smashed_activations, labels = client.forward_pass()
        
        # 2. Server completes forward pass and backprops up to cut layer
        grad_to_client, loss = server.train_step(smashed_activations, labels)
        epoch_loss += loss
        
        # 3. Client completes backward pass with given gradients
        client.backward_pass(grad_to_client)
        
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {epoch_loss/num_clients:.4f}")
    
    # 4. Global aggregation of client models (SplitFed V1)
    client_weights = [c.get_weights() for c in clients]
    global_weights = server.aggregate_client_models(client_weights)
    for client in clients:
        client.set_weights(global_weights)
        
print("Training simulation complete.")